# Proyecto - Predicción de Importaciones para Cencosud

Este notebook implementa un flujo de trabajo reproducible para estimar las importaciones de electrodomésticos en Chile con foco en apoyar la planificación logística y de abastecimiento de **Cencosud**.

## Objetivo
Construir un modelo predictivo que permita estimar el comportamiento futuro de las importaciones de electrodomésticos en Chile.

## Variables analizadas
- `valor_cif`
- `peso`
- `cantidad`

## Códigos HS considerados
- `8418`: Refrigeradores
- `8450`: Lavadoras
- `8516`: Microondas / hornos eléctricos
- `8528`: Televisores

## Horizonte de predicción
- `6 meses`

## Modelos a comparar
- ARIMA
- Regresión Lineal
- Random Forest
- XGBoost
- LightGBM

> **Nota importante**: Este notebook fue corregido para trabajar de forma coherente con columnas en **minúscula** y con los archivos:
> - `importaciones_hs_filtrado_estandarizado.csv`
> - `importaciones_hs_filtrado_raw.csv`

## BLOQUE 2 — Librerías a utilizar

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import os
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from statsmodels.tsa.arima.model import ARIMA

# Modelos opcionales
xgboost_disponible = True
lightgbm_disponible = True

try:
    from xgboost import XGBRegressor
except Exception:
    xgboost_disponible = False

try:
    from lightgbm import LGBMRegressor
except Exception:
    lightgbm_disponible = False

print("Librerías cargadas correctamente.")
print(f"XGBoost disponible: {xgboost_disponible}")
print(f"LightGBM disponible: {lightgbm_disponible}")

## BLOQUE 3 — Parámetros del proyecto

In [ ]:
# Rutas
REPO_DIR = "/content/MBD00010"
RAW_DATA_DIR = "/content/drive/MyDrive/Mod10/proyecto_cencosud_importaciones/data/raw/aduanas"
PROCESSED_DIR = "/content/drive/MyDrive/Mod10/proyecto_cencosud_importaciones/data/processed"

BASE_DIR = Path(r"C:\Users\roberto.sepulveda\OneDrive - udp.cl\Escritorio\Roberto\Magister 2025\MBD0010\Tarea_Final\Proyecto_Logistica_Cencosud")
DATA_DIR = BASE_DIR / "data" / "processed"
RESULTS_DIR = BASE_DIR / "results"
GRAPHICS_DIR = RESULTS_DIR / "graphics"
TABLES_DIR = RESULTS_DIR / "tables"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
GRAPHICS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

# Archivos
FILE_EST = DATA_DIR / "importaciones_hs_filtrado_estandarizado.csv"
FILE_RAW = DATA_DIR / "importaciones_hs_filtrado_raw.csv"

# Parámetros analíticos
HS_CODES_OBJETIVO = ["8418", "8450", "8516", "8528"]
VARIABLES_OBJETIVO = ["valor_cif", "peso", "cantidad"]
HORIZONTE_MESES = 6

print("Rutas configuradas correctamente.")
print("Archivo estandarizado:", FILE_EST)
print("Archivo raw:", FILE_RAW)

## BLOQUE 4 — Funciones auxiliares

In [ ]:
def leer_csv_seguro(path_obj):
    """
    Intenta leer un CSV con varias combinaciones comunes de encoding y separador.
    """
    intentos = [
        {"sep": ",", "encoding": "utf-8"},
        {"sep": ";", "encoding": "utf-8"},
        {"sep": ",", "encoding": "latin-1"},
        {"sep": ";", "encoding": "latin-1"},
    ]

    for intento in intentos:
        try:
            df_tmp = pd.read_csv(path_obj, sep=intento["sep"], encoding=intento["encoding"], low_memory=False)
            if df_tmp.shape[1] > 1:
                print(f"[OK] Lectura exitosa: {path_obj.name} | sep='{intento['sep']}' | encoding='{intento['encoding']}'")
                return df_tmp
        except Exception:
            continue

    raise ValueError(f"No fue posible leer el archivo: {path_obj}")


def estandarizar_columnas(df):
    df = df.copy()
    df.columns = [str(c).strip().lower() for c in df.columns]
    return df


def detectar_columna_fecha(df):
    candidatos = ["fectra", "fecha", "fecha_documento", "fecha_registro", "fec_tra", "fechadoc"]
    for c in candidatos:
        if c in df.columns:
            return c
    return None


def detectar_columna_codigo_arancel(df):
    candidatos = ["codigo_arancel", "aranc_nac", "codigo_hs", "partida", "hs", "codigo"]
    for c in candidatos:
        if c in df.columns:
            return c
    return None


def parsear_fecha_robusta(serie):
    """
    Intenta convertir fechas con diferentes formatos.
    """
    s = serie.copy()

    # intento general
    f1 = pd.to_datetime(s, errors="coerce", dayfirst=True)
    if f1.notna().sum() > 0:
        return f1

    # intento con solo dígitos YYYYMMDD
    s2 = s.astype(str).str.replace(r"[^0-9]", "", regex=True)
    f2 = pd.to_datetime(s2, format="%Y%m%d", errors="coerce")
    if f2.notna().sum() > 0:
        return f2

    # intento alternativo YYYY-MM-DD implícito
    f3 = pd.to_datetime(s.astype(str), errors="coerce")
    return f3


def reconstruir_hs4(df, col_hs4="hs4", col_codigo_arancel=None):
    """
    Reconstruye hs4 de forma robusta.
    Prioridad:
    1) hs4 existente
    2) codigo_arancel
    """
    df = df.copy()

    base = None

    if col_hs4 in df.columns:
        base = df[col_hs4].astype(str)

    if col_codigo_arancel and col_codigo_arancel in df.columns:
        base_alt = df[col_codigo_arancel].astype(str)

        # Si hs4 no sirve o está vacío, preferir codigo_arancel
        if base is None:
            base = base_alt
        else:
            base = np.where(
                base.astype(str).str.strip().isin(["", "nan", "none"]),
                base_alt,
                base
            )
            base = pd.Series(base, index=df.index)

    if base is None:
        df["hs4_limpio"] = np.nan
        return df

    hs4_limpio = (
        pd.Series(base, index=df.index)
        .astype(str)
        .str.strip()
        .str.replace(r"[^0-9]", "", regex=True)
        .str[:4]
    )

    df["hs4_limpio"] = hs4_limpio
    return df


def asegurar_numericas(df, cols):
    df = df.copy()
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
        else:
            df[c] = 0
    return df


def crear_features_temporales(df, target_col):
    """
    Crea variables simples de series de tiempo para modelos supervisados.
    """
    d = df.copy().sort_values("fecha_mes").reset_index(drop=True)
    d["t"] = np.arange(len(d))
    d["mes"] = d["fecha_mes"].dt.month
    d["anio"] = d["fecha_mes"].dt.year
    d["lag_1"] = d[target_col].shift(1)
    d["lag_2"] = d[target_col].shift(2)
    d["lag_3"] = d[target_col].shift(3)
    d["rolling_3"] = d[target_col].shift(1).rolling(3).mean()
    d["rolling_6"] = d[target_col].shift(1).rolling(6).mean()
    return d


def metricas_regresion(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    try:
        mape = np.mean(np.abs((y_true - y_pred) / np.where(y_true == 0, 1, y_true))) * 100
    except Exception:
        mape = np.nan
    r2 = r2_score(y_true, y_pred) if len(y_true) > 1 else np.nan
    return {
        "mae": mae,
        "rmse": rmse,
        "mape": mape,
        "r2": r2
    }


def entrenar_y_evaluar_supervisado(df_feat, target_col, nombre_modelo, modelo):
    d = df_feat.dropna().copy()

    if len(d) < 12:
        return None, None, f"No hay suficientes datos para {nombre_modelo}"

    features = ["t", "mes", "anio", "lag_1", "lag_2", "lag_3", "rolling_3", "rolling_6"]
    X = d[features]
    y = d[target_col]

    test_size = max(6, int(len(d) * 0.2))
    if len(d) <= test_size:
        return None, None, f"Datos insuficientes para particionar en {nombre_modelo}"

    X_train, X_test = X.iloc[:-test_size], X.iloc[-test_size:]
    y_train, y_test = y.iloc[:-test_size], y.iloc[-test_size:]

    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)

    mets = metricas_regresion(y_test, y_pred)
    mets["modelo"] = nombre_modelo

    pred_df = pd.DataFrame({
        "fecha_mes": d.iloc[-test_size:]["fecha_mes"].values,
        "y_real": y_test.values,
        "y_pred": y_pred
    })

    return mets, pred_df, None


def forecast_supervisado(df_feat, target_col, modelo):
    d = df_feat.copy().sort_values("fecha_mes").reset_index(drop=True)
    d_model = d.dropna().copy()

    if len(d_model) < 12:
        return None

    features = ["t", "mes", "anio", "lag_1", "lag_2", "lag_3", "rolling_3", "rolling_6"]
    X = d_model[features]
    y = d_model[target_col]
    modelo.fit(X, y)

    futuro = d.copy()
    ultima_fecha = futuro["fecha_mes"].max()

    for _ in range(HORIZONTE_MESES):
        nueva_fecha = ultima_fecha + pd.offsets.MonthBegin(1)
        nueva_fila = {
            "fecha_mes": nueva_fecha,
            target_col: np.nan
        }
        futuro = pd.concat([futuro, pd.DataFrame([nueva_fila])], ignore_index=True)

        futuro["t"] = np.arange(len(futuro))
        futuro["mes"] = futuro["fecha_mes"].dt.month
        futuro["anio"] = futuro["fecha_mes"].dt.year
        futuro["lag_1"] = futuro[target_col].shift(1)
        futuro["lag_2"] = futuro[target_col].shift(2)
        futuro["lag_3"] = futuro[target_col].shift(3)
        futuro["rolling_3"] = futuro[target_col].shift(1).rolling(3).mean()
        futuro["rolling_6"] = futuro[target_col].shift(1).rolling(6).mean()

        row_pred = futuro.iloc[[-1]][features]
        pred = modelo.predict(row_pred)[0]
        futuro.loc[futuro.index[-1], target_col] = pred

        ultima_fecha = nueva_fecha

    salida = futuro.tail(HORIZONTE_MESES)[["fecha_mes", target_col]].copy()
    salida.rename(columns={target_col: "forecast"}, inplace=True)
    return salida


def entrenar_y_evaluar_arima(df_ts, target_col):
    d = df_ts[["fecha_mes", target_col]].dropna().copy().sort_values("fecha_mes")
    if len(d) < 18:
        return None, None, "No hay suficientes datos para ARIMA"

    serie = d[target_col].astype(float).values
    test_size = max(6, int(len(d) * 0.2))

    train = serie[:-test_size]
    test = serie[-test_size:]
    fechas_test = d["fecha_mes"].iloc[-test_size:].values

    try:
        modelo = ARIMA(train, order=(1, 1, 1))
        ajuste = modelo.fit()
        pred = ajuste.forecast(steps=test_size)

        mets = metricas_regresion(test, pred)
        mets["modelo"] = "ARIMA"

        pred_df = pd.DataFrame({
            "fecha_mes": fechas_test,
            "y_real": test,
            "y_pred": pred
        })

        return mets, pred_df, None
    except Exception as e:
        return None, None, str(e)


def forecast_arima(df_ts, target_col):
    d = df_ts[["fecha_mes", target_col]].dropna().copy().sort_values("fecha_mes")
    if len(d) < 18:
        return None

    serie = d[target_col].astype(float).values

    try:
        modelo = ARIMA(serie, order=(1, 1, 1))
        ajuste = modelo.fit()
        pred = ajuste.forecast(steps=HORIZONTE_MESES)

        fechas_futuras = pd.date_range(
            start=d["fecha_mes"].max() + pd.offsets.MonthBegin(1),
            periods=HORIZONTE_MESES,
            freq="MS"
        )

        return pd.DataFrame({
            "fecha_mes": fechas_futuras,
            "forecast": pred
        })
    except Exception:
        return None


def guardar_tabla(df, nombre_archivo):
    ruta = TABLES_DIR / nombre_archivo
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"[OK] Tabla guardada: {ruta}")


def guardar_figura(nombre_archivo):
    ruta = GRAPHICS_DIR / nombre_archivo
    plt.savefig(ruta, bbox_inches="tight", dpi=150)
    print(f"[OK] Gráfico guardado: {ruta}")

## BLOQUE 5 — Carga de datos

In [ ]:
print("Cargando datasets...")

if not FILE_EST.exists():
    raise FileNotFoundError(f"No existe el archivo: {FILE_EST}")

if not FILE_RAW.exists():
    print("[WARNING] No se encontró el archivo raw. Se trabajará solo con el estandarizado.")

df_est = leer_csv_seguro(FILE_EST)
df_est = estandarizar_columnas(df_est)

print("\nShape df_est:", df_est.shape)
print("Columnas df_est:")
print(df_est.columns.tolist())

if FILE_RAW.exists():
    df_raw = leer_csv_seguro(FILE_RAW)
    df_raw = estandarizar_columnas(df_raw)
    print("\nShape df_raw:", df_raw.shape)
    print("Columnas df_raw:")
    print(df_raw.columns.tolist())
else:
    df_raw = None

## BLOQUE 6 — Selección del dataframe base

In [ ]:
# Usaremos el estandarizado como base principal
df = df_est.copy()

print("DataFrame base seleccionado: df_est")
print("Shape inicial de df:", df.shape)
display(df.head())

## BLOQUE 7 — Diagnóstico inicial de columnas clave

In [ ]:
print("Diagnóstico inicial...")

col_fecha = detectar_columna_fecha(df)
col_codigo_arancel = detectar_columna_codigo_arancel(df)

print("Columna fecha detectada:", col_fecha)
print("Columna código arancel detectada:", col_codigo_arancel)
print("¿Existe columna hs4?:", "hs4" in df.columns)

if col_fecha is None:
    raise ValueError("No fue posible detectar una columna de fecha.")

if col_codigo_arancel is None and "hs4" not in df.columns:
    raise ValueError("No fue posible detectar una columna para reconstruir hs4.")

## paso previo

In [ ]:
def parsear_fecha_robusta(serie):
    """
    Convierte fechas almacenadas como enteros o texto en formato:
    - 4042024  -> 04-04-2024
    - 2042024  -> 02-04-2024
    - 22042024 -> 22-04-2024

    Reglas:
    - 7 dígitos: DMMYYYY  -> se completa con 0 a la izquierda
    - 8 dígitos: DDMMYYYY
    """

    fechas = []

    for x in serie:
        if pd.isna(x):
            fechas.append(pd.NaT)
            continue

        s = str(x).strip()
        s = re.sub(r"[^0-9]", "", s)

        if len(s) == 7:
            s = "0" + s
        elif len(s) == 8:
            pass
        else:
            fechas.append(pd.NaT)
            continue

        dia = s[:2]
        mes = s[2:4]
        anio = s[4:8]

        fecha_str = f"{dia}-{mes}-{anio}"

        try:
            fecha = pd.Timestamp.strptime(fecha_str, "%d-%m-%Y")
        except Exception:
            try:
                fecha = pd.to_datetime(fecha_str, format="%d-%m-%Y", errors="coerce")
            except Exception:
                fecha = pd.NaT

        fechas.append(fecha)

    return pd.Series(fechas, index=serie.index)

ejemplo_fechas = pd.Series([4042024, 2042024, 22042024, 8042024, 1042024, 15042024, 29042024])
print(parsear_fecha_robusta(ejemplo_fechas))

## BLOQUE 8 — Limpieza y estandarización de variables clave

In [ ]:
print("Estandarizando variables clave...")

df_model = df.copy()

# Fecha robusta
df_model["fecha_original"] = df_model[col_fecha].astype(str)
df_model["fecha_dt"] = parsear_fecha_robusta(df_model[col_fecha])

# Reconstrucción de hs4
df_model = reconstruir_hs4(df_model, col_hs4="hs4", col_codigo_arancel=col_codigo_arancel)

# Variables numéricas
df_model = asegurar_numericas(df_model, VARIABLES_OBJETIVO)

print("Fechas válidas detectadas:", df_model["fecha_dt"].notna().sum(), "de", len(df_model))
print("\nValores únicos hs4_limpio (primeros 30):")
print(df_model["hs4_limpio"].dropna().astype(str).unique()[:30])

cols_mostrar = ["fecha_original", "fecha_dt", "hs4_limpio"]
if col_codigo_arancel is not None:
    cols_mostrar.insert(2, col_codigo_arancel)

display(df_model[cols_mostrar].head(20))

## BLOQUE 9 — Filtrado de HS objetivo y construcción de series mensuales

In [ ]:
print("Construyendo series mensuales...")

df_model = df_model.dropna(subset=["fecha_dt"]).copy()
df_model = df_model[df_model["hs4_limpio"].astype(str).str.len() == 4].copy()

print("Shape después de limpieza base:", df_model.shape)

print("\nTop 20 hs4_limpio antes de filtrar objetivo:")
print(df_model["hs4_limpio"].value_counts().head(20))

df_model = df_model[df_model["hs4_limpio"].isin(HS_CODES_OBJETIVO)].copy()

print("Shape después de filtrar HS objetivo:", df_model.shape)

if df_model.empty:
    print("[WARNING] No hay registros para los HS objetivo con la lógica actual.")
else:
    df_model["fecha_mes"] = df_model["fecha_dt"].dt.to_period("M").dt.to_timestamp()
    df_model["hs4"] = df_model["hs4_limpio"]

    monthly_total = (
        df_model.groupby("fecha_mes", as_index=False)[VARIABLES_OBJETIVO]
        .sum()
        .sort_values("fecha_mes")
        .reset_index(drop=True)
    )

    monthly_by_hs = (
        df_model.groupby(["hs4", "fecha_mes"], as_index=False)[VARIABLES_OBJETIVO]
        .sum()
        .sort_values(["hs4", "fecha_mes"])
        .reset_index(drop=True)
    )

    print("monthly_total shape:", monthly_total.shape)
    print("monthly_by_hs shape:", monthly_by_hs.shape)

    display(monthly_total.head(12))
    display(monthly_by_hs.head(12))

## BLOQUE 10 — Fallback automático si no encuentra HS objetivo

In [ ]:
# Este bloque solo se ejecuta si el filtro anterior quedó vacío
if df_model.empty:
    print("Intentando fallback automático desde codigo_arancel / raw...")

    if df_raw is not None:
        df_alt = df_raw.copy()
        col_fecha_alt = detectar_columna_fecha(df_alt)
        col_codigo_alt = detectar_columna_codigo_arancel(df_alt)

        print("Columna fecha raw:", col_fecha_alt)
        print("Columna código arancel raw:", col_codigo_alt)

        if col_fecha_alt is not None and (col_codigo_alt is not None or "hs4" in df_alt.columns):
            df_alt["fecha_original"] = df_alt[col_fecha_alt]
            df_alt["fectra_dt"] = parsear_fecha_robusta(df_alt[col_fecha_alt])
            df_alt = reconstruir_hs4(df_alt, col_hs4="hs4", col_codigo_arancel=col_codigo_alt)
            df_alt = asegurar_numericas(df_alt, VARIABLES_OBJETIVO)

            df_alt = df_alt.dropna(subset=["fectra_dt"]).copy()
            df_alt = df_alt[df_alt["hs4_limpio"].astype(str).str.len() == 4].copy()
            df_alt = df_alt[df_alt["hs4_limpio"].isin(HS_CODES_OBJETIVO)].copy()

            print("Shape fallback raw:", df_alt.shape)

            if not df_alt.empty:
                df_alt["fecha_mes"] = df_alt["fectra_dt"].dt.to_period("M").dt.to_timestamp()
                df_alt["hs4"] = df_alt["hs4_limpio"]

                monthly_total = (
                    df_alt.groupby("fecha_mes", as_index=False)[VARIABLES_OBJETIVO]
                    .sum()
                    .sort_values("fecha_mes")
                    .reset_index(drop=True)
                )

                monthly_by_hs = (
                    df_alt.groupby(["hs4", "fecha_mes"], as_index=False)[VARIABLES_OBJETIVO]
                    .sum()
                    .sort_values(["hs4", "fecha_mes"])
                    .reset_index(drop=True)
                )

                df_model = df_alt.copy()

                print("monthly_total shape:", monthly_total.shape)
                print("monthly_by_hs shape:", monthly_by_hs.shape)

                display(monthly_total.head(12))
                display(monthly_by_hs.head(12))
            else:
                print("[ERROR] Tampoco fue posible construir series desde raw.")
        else:
            print("[ERROR] El raw no tiene columnas suficientes.")

## BLOQUE 11 — Exportación de datasets limpios

In [ ]:
if "monthly_total" in globals() and "monthly_by_hs" in globals():
    guardar_tabla(df_model, "dataset_final_modelado.csv")
    guardar_tabla(monthly_total, "dataset_mensual_total.csv")
    guardar_tabla(monthly_by_hs, "dataset_mensual_por_hs.csv")
else:
    print("[WARNING] No se generaron datasets mensuales.")

In [ ]:
if "monthly_total" in globals():
    print("Resumen estadístico mensual total:")
    display(monthly_total.describe())

    plt.figure(figsize=(12, 5))
    plt.plot(monthly_total["fecha_mes"], monthly_total["valor_cif"])
    plt.title("Serie mensual total - Valor CIF")
    plt.xlabel("Fecha")
    plt.ylabel("Valor CIF")
    plt.xticks(rotation=45)
    plt.tight_layout()
    guardar_figura("serie_mensual_valor_cif_total.png")
    plt.show()

    plt.figure(figsize=(12, 5))
    plt.plot(monthly_total["fecha_mes"], monthly_total["peso"])
    plt.title("Serie mensual total - Peso")
    plt.xlabel("Fecha")
    plt.ylabel("Peso")
    plt.xticks(rotation=45)
    plt.tight_layout()
    guardar_figura("serie_mensual_peso_total.png")
    plt.show()

    plt.figure(figsize=(12, 5))
    plt.plot(monthly_total["fecha_mes"], monthly_total["cantidad"])
    plt.title("Serie mensual total - Cantidad")
    plt.xlabel("Fecha")
    plt.ylabel("Cantidad")
    plt.xticks(rotation=45)
    plt.tight_layout()
    guardar_figura("serie_mensual_cantidad_total.png")
    plt.show()
else:
    print("[WARNING] No existe monthly_total.")

In [ ]:
if "monthly_by_hs" in globals():
    for variable in VARIABLES_OBJETIVO:
        plt.figure(figsize=(12, 6))
        for hs in sorted(monthly_by_hs["hs4"].unique()):
            temp = monthly_by_hs[monthly_by_hs["hs4"] == hs]
            plt.plot(temp["fecha_mes"], temp[variable], label=hs)

        plt.title(f"Serie mensual por HS - {variable}")
        plt.xlabel("Fecha")
        plt.ylabel(variable)
        plt.legend()
        plt.xticks(rotation=45)
        plt.tight_layout()
        guardar_figura(f"serie_por_hs_{variable}.png")
        plt.show()
else:
    print("[WARNING] No existe monthly_by_hs.")

In [ ]:
resultados_metricas = []
resultados_predicciones = {}
resultados_forecast = {}

if "monthly_total" in globals():
    print("Iniciando modelado...")
else:
    print("[ERROR] No existe monthly_total; no se puede modelar.")